# Crypto trend following on synthetic fixtures

A worked example of the `tf-trend[crypto]` extension, runnable offline: build a
7-day synthetic universe with staggered listings, run the 1/3/12-month
time-series momentum preset, and challenge the result with the falsification
suite. Swap the synthetic frame for your own data (see `docs/CRYPTO_DATA.md`)
before drawing any conclusion; synthetic prices are generated, not real.

In [ ]:
import tempfile

import pandas as pd

from tf.data.synthetic import generate_synthetic_prices
from tf.engine.backtester import Backtester

prices = generate_synthetic_prices(
    ["BTC", "ETH", "SOL"],
    "2018-01-01",
    "2024-12-31",
    freq="D",              # crypto trades every calendar day
    common_factor=0.6,     # crypto pairs are strongly correlated
    seed=11,
    start_offsets={"SOL": 900},  # SOL lists ~2.5 years in
)
prices.plot(logy=True, figsize=(9, 4), title="Synthetic 7-day reference prices");

In [ ]:
universe = [
    {"symbol": s, "sector": "Crypto", "point_value": 1.0, "contract_step": 1e-4}
    for s in ("BTC", "ETH", "SOL")
]

config = {
    "data": {"calendar": "CRYPTO_DAILY", "allow_partial_history": True},
    "universe": {
        "eligibility": {
            "min_history": "365D",
            "entry_lag": "30D",
            "evaluation_frequency": "monthly",
        }
    },
    "backtest": {
        "start": "2018-01-01",
        "end": "2024-12-31",
        "starting_nav": 1_000_000.0,
        "results_dir": tempfile.mkdtemp(),
    },
    "signals": {"preset": "tsmom_1_3_12", "direction": "long_short"},
    "risk": {
        "periods_per_year": 365,   # 7-day calendar; 252 understates vol ~17%
        "target_portfolio_vol": 0.10,
        "min_vol_periods": 90,
        "max_asset_weight": 0.5,
    },
    "execution": {
        "adv_limit_pct": 0.2,
        "adv_contracts": {"BTC": 20000, "ETH": 20000, "SOL": 20000},
        "impact": {"k": 0.02, "alpha": 0.5},
        "min_slippage_ticks": 0.5,
        "tick_value": 0.01,
        "spread_bps": {"BTC": 1.0, "ETH": 1.5, "SOL": 4.0},
    },
}

backtester = Backtester(prices, universe, config)
result = backtester.run()
result.nav.plot(figsize=(9, 4), title="Strategy NAV");

In [ ]:
from tf.eval.metrics import performance_summary

# periods_per_year must match the basis positions were sized on.
performance_summary(result.nav, trades=result.trades, periods_per_year=365)

## Point-in-time eligibility

SOL lists mid-window, so the mask keeps it out until it has a year of history
plus the entry lag. The data-span honesty panel makes the thin early universe
visible instead of implied.

In [ ]:
from tf.data.eligibility import eligibility_summary, thin_universe_fraction

mask = result.eligibility_mask
display(eligibility_summary(mask))
print(f"fraction of days with fewer than 3 eligible assets: "
      f"{thin_universe_fraction(mask, minimum=3):.1%}")

## Falsify it

The suite compares the strategy against benchmarks that require no skill,
re-runs it at higher costs and later signals, separates the trend signal's
contribution from volatility scaling's, and evaluates every non-identity sign
flip of the signal as a placebo. Read the notes: the report says when its own
tests cannot bite.

In [ ]:
from tf.eval.falsification import run_falsification

report = run_falsification(backtester, seed=0)
for note in report.notes:
    print("NOTE:", note, "\n")
report.variants[["CAGR", "Volatility", "Sharpe", "Max Drawdown"]].round(3)

In [ ]:
report.vol_scaling_2x2.round(3)

In [ ]:
report.placebo.round(3)

In [ ]:
report.cost_stress[["CAGR", "Sharpe", "Turnover (ann.)"]].round(3)

If the strategy's row does not stand out from these tables, that is the
finding. A result that only survives at 1x assumed costs, or that sits inside
the placebo's sign-flip distribution, is an assumption wearing a Sharpe ratio.